# Robot Calibration Editor
This notebook loads the existing calibration from `data/my_robot_calibration.json`, shows the current settings, and lets you click a value to edit it with the existing robot picker window.

In [1]:
import tkinter as tk
from tkinter import ttk

from robot import SerialPrinter, RobotCalibrator
from datatypes import RobotCalibration

In [2]:
printer = SerialPrinter()
printer.connect()

Connecting to GRBL on 0...
Connecting to GRBL on 1...
Connecting to GRBL on 2...
Connecting to GRBL on 3...
Connecting to GRBL on 4...
Connecting to GRBL on 5...
Waiting...
Connected successfully.


In [3]:
from pathlib import Path

calibration_path = Path("data/my_robot_calibration.json")
my_robot_calibration = RobotCalibration.load(calibration_path.as_posix())


def format_point(point):
    return f"({point[0]:.1f}, {point[1]:.1f}, {point[2]:.1f})"


def edit_point(query, current_value, ask_z=True, master=None):
    updated_value, _ = RobotCalibrator(
        printer,
        initial_position=current_value,
    ).input_position(
        query=query,
        color=False,
        ask_z=ask_z,
        master=master,
    )
    return updated_value

In [4]:
def launch_calibration_editor():
    root = tk.Tk()
    root.title("Robot Calibration Editor")
    root.geometry("980x760")

    container = ttk.Frame(root)
    container.pack(fill=tk.BOTH, expand=True)

    editor_canvas = tk.Canvas(container, highlightthickness=0)
    editor_scrollbar = ttk.Scrollbar(container, orient=tk.VERTICAL, command=editor_canvas.yview)
    editor_scrollable = ttk.Frame(editor_canvas, padding=16)
    editor_window = editor_canvas.create_window((0, 0), window=editor_scrollable, anchor="nw")
    editor_canvas.configure(yscrollcommand=editor_scrollbar.set)

    def update_scrollregion(event=None):
        if event is not None:
            event.widget
        editor_canvas.configure(scrollregion=editor_canvas.bbox("all"))

    def sync_scrollable_width(event):
        editor_canvas.itemconfigure(editor_window, width=event.width)

    def on_mousewheel(event):
        editor_canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")

    editor_scrollable.bind("<Configure>", update_scrollregion)
    editor_canvas.bind("<Configure>", sync_scrollable_width)
    editor_canvas.bind_all("<MouseWheel>", on_mousewheel)
    editor_canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
    editor_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

    main = editor_scrollable

    status_var = tk.StringVar(value=f"Loaded {calibration_path.as_posix()}")

    point_vars = {
        name: tk.StringVar()
        for name in (
            "top_left",
            "top_right",
            "bottom_left",
            "bottom_right",
            "water_reservoir",
        )
    }
    safe_height_var = tk.StringVar()

    palette_frame = ttk.LabelFrame(main, text="Color Palette", padding=12)
    palette_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 12))

    def persist_calibration(message):
        my_robot_calibration.dump(calibration_path.as_posix())
        status_var.set(message)

    def refresh():
        point_vars["top_left"].set(format_point(my_robot_calibration.top_left))
        point_vars["top_right"].set(format_point(my_robot_calibration.top_right))
        point_vars["bottom_left"].set(format_point(my_robot_calibration.bottom_left))
        point_vars["bottom_right"].set(format_point(my_robot_calibration.bottom_right))
        point_vars["water_reservoir"].set(format_point(my_robot_calibration.water_reservoir))
        safe_height_var.set(f"{float(my_robot_calibration.safe_height):.1f}")
        rebuild_palette_rows()
        root.update_idletasks()

    def open_picker(query, current_value, ask_z=True):
        root.withdraw()
        try:
            return edit_point(query, current_value, ask_z=ask_z, master=root)
        finally:
            if root.winfo_exists():
                root.deiconify()

    def edit_canvas_point(attribute, label):
        updated_value = open_picker(label, getattr(my_robot_calibration, attribute), ask_z=True)
        setattr(my_robot_calibration, attribute, tuple(updated_value))
        persist_calibration(f"Updated {label} and auto-saved")
        refresh()

    def edit_water_reservoir():
        updated_value = open_picker("Water Reservoir", my_robot_calibration.water_reservoir, ask_z=True)
        my_robot_calibration.water_reservoir = tuple(updated_value)
        persist_calibration("Updated Water Reservoir and auto-saved")
        refresh()

    def edit_safe_height():
        updated_value = open_picker(
            "Safe Height",
            (0.0, 0.0, float(my_robot_calibration.safe_height)),
            ask_z=True,
        )
        my_robot_calibration.safe_height = float(updated_value[2])
        persist_calibration("Updated Safe Height and auto-saved")
        refresh()

    def edit_palette_entry(index):
        entry = my_robot_calibration.color_palette.color_positions[index]
        updated_position, updated_color = RobotCalibrator(
            printer,
            initial_position=tuple(entry["position"]),
        ).input_position(
            query=f"Color {index + 1}",
            color=True,
            initial_color=entry["color"],
            master=root,
        )
        entry["position"] = tuple(updated_position)
        entry["color"] = updated_color or entry["color"]
        persist_calibration(f"Updated Color {index + 1} and auto-saved")
        refresh()

    def add_palette_color():
        palette_entries = my_robot_calibration.color_palette.color_positions
        default_position = palette_entries[-1]["position"] if palette_entries else my_robot_calibration.top_left
        new_position, new_color = RobotCalibrator(
            printer,
            initial_position=tuple(default_position),
        ).input_position(
            query="New Color",
            color=True,
            initial_color=palette_entries[-1]["color"] if palette_entries else "#FFFFFF",
            master=root,
        )
        my_robot_calibration.color_palette.add_color(tuple(new_position), new_color)
        persist_calibration("Added new color and auto-saved")
        refresh()

    def remove_palette_color():
        palette_entries = my_robot_calibration.color_palette.color_positions
        if palette_entries:
            palette_entries.pop()
            persist_calibration("Removed last color and auto-saved")
            refresh()
        else:
            status_var.set("No colors to remove")

    def reload_from_disk():
        loaded_calibration = RobotCalibration.load(calibration_path.as_posix())
        my_robot_calibration.top_left = loaded_calibration.top_left
        my_robot_calibration.top_right = loaded_calibration.top_right
        my_robot_calibration.bottom_left = loaded_calibration.bottom_left
        my_robot_calibration.bottom_right = loaded_calibration.bottom_right
        my_robot_calibration.water_reservoir = loaded_calibration.water_reservoir
        my_robot_calibration.safe_height = loaded_calibration.safe_height
        my_robot_calibration.color_palette.load(loaded_calibration.color_palette.dump())
        refresh()
        status_var.set(f"Reloaded {calibration_path.as_posix()}")

    header = ttk.Label(
        main,
        text="Click a value to edit it with the live robot picker.",
        font=("Helvetica", 12, "bold"),
        padding=(0, 0, 0, 12),
    )
    header.pack(anchor=tk.W)

    canvas_frame = ttk.LabelFrame(main, text="Canvas Positions", padding=12)

    def add_point_row(parent, label, value_var, command):
        row = ttk.Frame(parent)
        row.pack(fill=tk.X, pady=4)
        ttk.Label(row, text=label, width=18).pack(side=tk.LEFT)
        ttk.Button(row, textvariable=value_var, command=command).pack(side=tk.LEFT, fill=tk.X, expand=True)

    add_point_row(
        canvas_frame,
        "Top Left",
        point_vars["top_left"],
        lambda: edit_canvas_point("top_left", "Top Left"),
    )
    add_point_row(
        canvas_frame,
        "Top Right",
        point_vars["top_right"],
        lambda: edit_canvas_point("top_right", "Top Right"),
    )
    add_point_row(
        canvas_frame,
        "Bottom Left",
        point_vars["bottom_left"],
        lambda: edit_canvas_point("bottom_left", "Bottom Left"),
    )
    add_point_row(
        canvas_frame,
        "Bottom Right",
        point_vars["bottom_right"],
        lambda: edit_canvas_point("bottom_right", "Bottom Right"),
    )
    add_point_row(
        canvas_frame,
        "Water Reservoir",
        point_vars["water_reservoir"],
        edit_water_reservoir,
    )
    add_point_row(
        canvas_frame,
        "Safe Height",
        safe_height_var,
        edit_safe_height,
    )

    canvas_frame.pack(fill=tk.X, pady=(0, 12))

    palette_controls = ttk.Frame(main)
    palette_controls.pack(fill=tk.X, pady=(0, 8))
    ttk.Button(palette_controls, text="Add Color", command=add_palette_color).pack(side=tk.LEFT)
    ttk.Button(palette_controls, text="Remove Last Color", command=remove_palette_color).pack(side=tk.LEFT, padx=(8, 0))

    palette_rows = ttk.Frame(palette_frame)
    palette_rows.pack(fill=tk.BOTH, expand=True)

    def rebuild_palette_rows():
        for child in palette_rows.winfo_children():
            child.destroy()

        palette_entries = my_robot_calibration.color_palette.color_positions
        if not palette_entries:
            ttk.Label(palette_rows, text="No colors are defined yet.").pack(anchor=tk.W)
            return

        for index, entry in enumerate(palette_entries):
            row = ttk.Frame(palette_rows)
            row.pack(fill=tk.X, pady=4)
            ttk.Label(row, text=f"Color {index + 1}", width=18).pack(side=tk.LEFT)
            ttk.Label(row, text=format_point(tuple(entry["position"])), width=24).pack(side=tk.LEFT, padx=(0, 8))
            ttk.Label(row, text=str(entry["color"]), width=12).pack(side=tk.LEFT, padx=(0, 8))
            ttk.Button(row, text="Edit", command=lambda index=index: edit_palette_entry(index)).pack(side=tk.LEFT)

    control_frame = ttk.Frame(main)
    control_frame.pack(fill=tk.X)
    ttk.Button(control_frame, text="Reload", command=reload_from_disk).pack(side=tk.LEFT)

    status_label = ttk.Label(main, textvariable=status_var, padding=(0, 12, 0, 0))
    status_label.pack(anchor=tk.W)

    refresh()
    root.mainloop()


launch_calibration_editor()